#1 - Verificar versões

In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA indisponível, rodando em CPU")

torch: 2.6.0+cu124
CUDA runtime: 12.4
GPU: Tesla T4


#2 - Instalar só as dependências faltantes para QLoRA


In [ ]:
!pip install -q \
  transformers \
  datasets \
  accelerate \
  peft \
  bitsandbytes \
  trl \
  safetensors \
  huggingface-hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.3/366.3 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 121.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# 3 - importações e configuração inicial

In [ ]:
import random
import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer

import logging

# seed pra tornar o experimento reprodutível
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(1234)

4.1 - login e carregamento do tokenizer com token de acesso

In [ ]:
import os
from huggingface_hub import login
from transformers import AutoTokenizer

os.environ["HF_TOKEN"] = HF_KEY

login(token=os.environ["HF_TOKEN"])

MODEL_NAME = "mistralai/Mistral-7B-v0.1"
token = os.environ["HF_TOKEN"]

# carregando tokenizer usando o token autenticado
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_auth_token=token,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# teste rápido
print(tokenizer("Olá, mundo!")["input_ids"])

# 5 - Configuração do tokenizer 2


In [ ]:
import os
from transformers import AutoTokenizer

MODEL_NAME = "mistralai/Mistral-7B-v0.1"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_auth_token=True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 6 - Carregar dataset e preparar DataCollator


In [ ]:
from datasets import load_dataset
from transformers import DataCollatorForLanguageModeling
import os

token = os.environ.get("HF_TOKEN")

dataset = load_dataset("databricks/databricks-dolly-15k", split="train", token=token)
# teste rápido, só um subset:
# dataset = dataset.select(range(1000))

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
    pad_to_multiple_of=8
)

print(f"Dataset carregado: {dataset}")

#  7 - quantização e carregamento do modelo em 4-bit


In [ ]:
import os
import torch
from transformers import BitsAndBytesConfig, AutoModelForCausalLM

def get_quantization_config():
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=dtype,
        bnb_4bit_use_double_quant=True,
    )

def load_quantized_base(model_name: str, token: str):
    q_config = get_quantization_config()
    return AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=q_config,
        device_map="auto",
        trust_remote_code=True,
        use_auth_token=token
    )

# carregando o modelo quantizado
base_model = load_quantized_base(MODEL_NAME, os.environ["HF_TOKEN"])
base_model.config.use_cache = False
base_model.config.pretraining_tp = 1

print("Modelo base carregado em 4-bit e também pronto pra LoRA")

# 8 - preparando o modelo para k-bit training e ajustar com LoRA


In [ ]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

model = prepare_model_for_kbit_training(base_model)

# configurando o lora
peft_config = LoraConfig(
    task_type="CAUSAL_LM",
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj",
                    "v_proj"],
    bias="none"
)

model = get_peft_model(model, peft_config)
model.config.pad_token_id = tokenizer.pad_token_id
model.print_trainable_parameters()

trainable params: 6,815,744 || all params: 7,248,547,840 || trainable%: 0.0940


#9 - Teste rápido do fine-tuning com lora

In [ ]:
from trl import SFTConfig, SFTTrainer

def format_instruction(sample: dict) -> str:
    instr = sample.get("instruction", "")
    ctx   = sample.get("context")
    resp  = sample.get("response", "")
    prompt = f"### Instrução:\n{instr}\n\n"
    if ctx:
        prompt += f"### Contexto:\n{ctx}\n\n"
    prompt += f"### Resposta:\n{resp}"
    return prompt


# fazendo pré-processamento p/ garantir que o trainer tá usando o formatting_func
train_subset = dataset.select(range(64))
eval_subset  = dataset.select(range(64, 80))

# teste rápido de 10 steps
training_args = SFTConfig(
    output_dir="test_lora",
    num_train_epochs=1,
    max_steps=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=2,
    eval_strategy="steps",
    eval_steps=5,
    save_strategy="no",
    fp16=True,
)

# montando o trainer usando formatting_func
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_subset,
    eval_dataset=eval_subset,
    peft_config=peft_config,
    data_collator=data_collator,
    processing_class=tokenizer,
    formatting_func=format_instruction,
)

# executa o treino curto
trainer.train()

# 10 - testando geração com o modelo lora treinado


In [ ]:
model.eval()

instr = "Explique a teoria da relatividade de forma simples."
prompt = format_instruction({
    "instruction": instr,
    "context": "",
    "response": ""
})

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    padding=True,
    truncation=True
).to(model.device)

gen_ids = model.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=128,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

output = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
print(output)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


### Instrução:
Explique a teoria da relatividade de forma simples.

### Resposta:
A teoria da relatividade é uma teoria que modifica a teoria de Newton sobre o movimento e o tempo. De acordo com esta teoria, o tempo e o espaço são interdependentes, e o tempo passa mais lentamente para as pessoas que se movem rapidamente. Além disso, a teoria da relatividade afirma que o tempo e o espaço são diferentes para diferentes pessoas, dependendo de como elas se movem.

A teoria da relatividade foi desenvolvida por


# 10.1 - Métricas

In [ ]:
!pip install -q rouge_score

  Preparing metadata (setup.py) ... done


In [ ]:
import torch
import numpy as np
from rouge_score import rouge_scorer

# Função de perplexidade
def compute_perplexity(model, tokenizer, ds, num_samples=None):
    model.eval()
    losses = []
    subset = ds if num_samples is None else ds.select(range(num_samples))
    for ex in subset:
        prompt = format_instruction(ex)
        enc = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            padding=True
        ).to(model.device)
        with torch.no_grad():
            out = model(**enc, labels=enc["input_ids"])
        losses.append(out.loss.item())
    return float(np.exp(np.mean(losses)))

eval_dataset = dataset.select(range(64, 80))

# perplexidade
ppl = compute_perplexity(model, tokenizer, eval_dataset)
print(f"Perplexidade média: {ppl:.2f}")

# gerando previsões e coleta referências
preds, refs = [], []
for ex in eval_dataset:
    prompt = format_instruction({
        "instruction": ex["instruction"],
        "context": ex.get("context",""),
        "response": ""
    })
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True).to(model.device)
    gen_ids = model.generate(
        enc["input_ids"],
        attention_mask=enc["attention_mask"],
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
    text = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
    gen_resp = text.split("### Resposta:")[-1].strip()
    preds.append(gen_resp)
    refs.append(ex["response"])

# ROUGE-1 e ROUGE-L
scorer = rouge_scorer.RougeScorer(["rouge1","rougeL"], use_stemmer=True)
scores = [scorer.score(r, p) for r, p in zip(refs, preds)]

rouge1 = np.mean([s["rouge1"].fmeasure for s in scores])
rougeL = np.mean([s["rougeL"].fmeasure for s in scores])

print(f"ROUGE-1 (f1): {rouge1:.3f}")
print(f"ROUGE-L (f1): {rougeL:.3f}")

Perplexidade média: 4.00
ROUGE-1 (f1): 0.364
ROUGE-L (f1): 0.291
